# Getting Started with Calling LLM APIs using Gemini and LangChain 🦜🔗

In this notebook, you will learn how to use LLM APIs via LangChain. As an example, we will use Google's Gemini API. By the end of this notebook, you will know how to make API calls using LangChain and understand why we do it this way.

## ⚙️ Setup

👉 Run the cell below to load the environment variables from the `.env` file we created during the setup phase:

In [1]:
%pip install python-dotenv


[notice] A new release of pip is available: 25.3 -> 26.0
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
from dotenv import load_dotenv

load_dotenv() # Load environment variables from .env file

True

👉 Is the output of the cell `True`? Great! That means we have successfully set up a `GOOGLE_API_KEY` environment variable that will be used to authenticate with the Gemini API.

If not, ask for help.

## Making a Simple API Call

In this notebook, we will show how to:
1. Make an API call using Google's own client library.
2. Do the same thing using LangChain.

## Using the Google Generative AI Library

In [3]:
%pip install google-genai


[notice] A new release of pip is available: 25.3 -> 26.0
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
from google import genai

In [5]:
client = genai.Client()

response = client.models.generate_content(
    model="gemini-2.5-flash-lite",
    contents="What is the capital of France?",
)

Let’s take a look at the `response` object.

In [6]:
response.candidates[0].content.parts[0].text

'The capital of France is **Paris**.'

Do you see how to get the actual answer?

Fortunately, we can just use the `.text` property to get the answer directly. Try it.

In [7]:
response.text

'The capital of France is **Paris**.'

Gemini returns its answers in Markdown format. Let’s take advantage of that!

In [8]:
from IPython.display import Markdown
Markdown(response.text)

The capital of France is **Paris**.

You can also change the generation parameters. Here is how you do that using `google.genai`:

In [9]:
from google import genai
from google.genai import types # We need to import types for the config

client = genai.Client()

response = client.models.generate_content(
    model="gemini-2.5-flash-lite",
    contents="Write a social media post about how much you're learning about transformers.",
    config=types.GenerateContentConfig(
        max_output_tokens=200,
        temperature=1.0
    )
)

In [10]:
Markdown(response.text)

Here are a few options for a social media post about learning transformers, ranging in tone and detail. Choose the one that best fits your style!

**Option 1: Enthusiastic & Brief**

> Dive deep into the fascinating world of Transformers! 🤖 My brain is buzzing with all the new things I'm learning about how these models work. So cool to see the power of attention mechanisms at play. #Transformers #AI #MachineLearning #DeepLearning #NLP

**Option 2: Slightly More Detailed & Personal**

> I'm on a serious learning journey with Transformer models right now, and WOW! 🤯 The concepts of self-attention and positional encoding are truly mind-bending, but in the best way possible. It's amazing how they've revolutionized NLP and beyond. Definitely keeping this momentum going! #AIJourney #TransformerModels #NaturalLanguageProcessing #TechEducation

**Option 3: A Bit More Technical (but still approachable)**

>

Great. But what if you want to try a different API, such as OpenAI or Anthropic?

You would need to read their documentation and rewrite your code to use their API instead. It would be similar, but not exactly the same.

Fortunately, we have LangChain!

## Using LangChain 🦜🔗

Why would you use LangChain?

1. **Model-Independent Code**

   LangChain provides abstractions that let you switch between different LLM providers (Google, OpenAI, Anthropic, etc.) with minimal code changes. If you write directly against a specific provider’s API, changing providers later usually requires significant refactoring.

2. **Unified Interface**

   LangChain standardizes how you interact with different LLM providers, offering consistent methods and response formats regardless of the underlying API.

3. **Composable Components**

   LangChain’s chains and pipeline architecture make it easy to build complex workflows that combine prompts, memory, and tools without having to build all the plumbing yourself.

4. **Built-in Utilities**

   LangChain includes helpers like output parsers, prompt templates, and other tools you would otherwise need to implement on your own.

Go to the [list of LangChain chat integrations](https://docs.langchain.com/oss/python/integrations/chat) and look through the integrations. Can you find your favorite LLM provider there?

We don’t want to use `chat_models.ChatGoogleGenerativeAI` in our code because it is specific to Gemini. If we decide to change the LLM later, we would also have to change how we initialize the model. Fortunately, LangChain provides a more general way to initialize a model.

Let’s use Gemini again, but this time through LangChain’s generic Chat Models interface.

👉 Go to the [LangChain "Models" documentation](https://docs.langchain.com/oss/python/langchain/models) and find out how to initialize a chat model using Gemini.

Tips:
1. Jump straight to the "Basic Usage" section.
2. By selecting the model you want to use, you can quickly see the relevant documentation.

In [14]:
%pip install langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [langchain-google-genai]

[notice] A new release of pip is available: 25.3 -> 26.0
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [15]:
from langchain.chat_models import init_chat_model

model = init_chat_model("gemini-2.5-flash-lite", model_provider="google_genai")

The most basic way to use the model is simply to call the `.invoke()` method:

In [23]:
response = model.invoke("What is the capital of France?")

Let’s inspect the response. We’ll use `pprint()` to nicely print its `__dict__`, which includes all of the object’s attributes and methods.

In [24]:
from pprint import pprint
pprint(response.__dict__)

{'additional_kwargs': {},
 'content': 'The capital of France is **Paris**.',
 'id': 'lc_run--019c23cd-35e2-72a2-b357-fb00c3153473-0',
 'invalid_tool_calls': [],
 'name': None,
 'response_metadata': {'finish_reason': 'STOP',
                       'model_name': 'gemini-2.5-flash-lite',
                       'model_provider': 'google_genai',
                       'safety_ratings': []},
 'tool_calls': [],
 'type': 'ai',
 'usage_metadata': {'input_token_details': {'cache_read': 0},
                    'input_tokens': 7,
                    'output_tokens': 8,
                    'total_tokens': 15}}


Extract the answer and display it. Remember that it is in Markdown format, so you can render it nicely.

In [25]:
Markdown(response.content)

The capital of France is **Paris**.

You can check the model’s temperature value by accessing the `.temperature` attribute. Give it a try:

In [27]:
model.temperature, model.max_output_tokens

(0.7, None)

Before using the model, we can also configure the generation parameters by assigning new values to its attributes.

Try to code the equivalent of what we did earlier with Google’s library to write a social media post.

> _Note_: Normally, we should be able to set the model’s `max_output_tokens` value (either when initializing the model or by changing the attribute later). However, the current version of `langchain_google_genai` (4.1.1) has a [bug](https://github.com/langchain-ai/langchain-google/issues/1454) and this does not work. The workaround? Set `max_output_tokens` as a parameter to the `.invoke()` method.

In [28]:
# Set the maximum number of output tokens to 200

model.max_output_tokens = 200

# Set the temperature to 1.0

model.temperature = 1.0

# Generate a response with the new settings

response = model.invoke("Write a social media post about how much you're learning about transformers.")

# Display the response

Markdown(response.content)


Here are a few options for your social media post, choose the one that best fits your style and platform!

**Option 1: Enthusiastic & Slightly Technical**

🤯 Officially deep-diving into the world of #Transformers and my brain is BUZZING! The sheer power and versatility of these neural network architectures are mind-blowing. From #NLP tasks like translation and text generation to their growing applications in #ComputerVision, I'm constantly amazed by what's possible. Learning so much about attention mechanisms, embeddings, and the elegance of self-attention. This is seriously changing the game! 🚀 #MachineLearning #DeepLearning #AI #Tech

**Option 2: Casual & Relatable**

Okay, who else is getting lost (in a good way!) in the world of #Transformers lately? 🙋‍♀️ This whole "attention is all you need" concept is proving to be a serious revelation. My understanding of how these models process information is evolving so fast

What’s the advantage of this? This LangChain Chat Model can support many other APIs.

To switch to another model, the only things you need to change are:
1. Obtain an API key for the other model and define it in your code.
2. Change the model and provider when initializing the model.

### Multiple Messages

Using the `.invoke()` function with just a single message can be a bit limiting.

You can instead provide multiple messages, such as:
- `SystemMessage` or system messages: to tell the model how it should behave
- `HumanMessage` or user messages: input coming from the user
- `AIMessage` or assistant messages: responses coming from the model

Let’s build a social media writer.

We’ll send a system message that explains how the model should behave. Then, in the user message, we can limit ourselves to just giving the topic it should write about.

To see how to do this, check out the [LangChain "Messages" documentation](https://docs.langchain.com/oss/python/langchain/messages).

Need inspiration for the system message? Here is a basic instruction to get you started:

```python
"""You are a creative social media writer who composes posts for a Generative AI student.
Your posts always include wordplay and a clear call to action.
Your posts are at most 200 characters long.
You always use emojis.
"""
```

In [33]:
# Import the necessary classes

from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage, AIMessage, SystemMessage

# Create a list of messages

messages = [
    SystemMessage(content="You are a creative social media writer who composes posts for a Generative AI student. Your posts always include wordplay and a clear call to action. Your posts are at most 200 characters long. You always use emojis."),
    HumanMessage(content="Write a social media post about how much you're learning about transformers."),
]

# Generate a response using the list of messages

response = model.invoke(messages)

# Display the response

Markdown(response.content)

I'm totally *transformed* by all I'm learning about transformers! 🤓 Ready to *generate* some new knowledge? Follow for more AI insights! ✨ #AI #Transformers #DeepLearning

🏁 Congratulations! You are now comfortable writing basic prompts with multiple messages using LangChain.